# M8 · Lab 1 — Build & Run the Truck Delay SageMaker Pipeline
### Module 8 (Capstone) — Full Automation | Spine: Truck Delay Classification

| | |
|---|---|
| **Duration** | 100 min (incl. ~15 min concept pre-read) · **Difficulty** Advanced · **Tier 3** (SageMaker Pipelines — first encounter) |
| **Tools** | SageMaker Python SDK + **AWS CLI / boto3**, the real Truck Delay data |
| **Prerequisite** | M8 `instructor_setup` (S3 bucket); the step code in `M8_Lab_1_Pipeline_Steps/` (ships ready) |
| **Builds toward** | Lab 2 (Lambda lands data) + Lab 3 (EventBridge triggers this pipeline) |

## 💻 Where to run this
On the **same SageMaker notebook instance** from M6/M7 (3rd day; auto-stops overnight). It **must** run here so
`get_execution_role()` resolves and the pipeline jobs inherit the instance's role. The pipeline's Processing/Training jobs
are **ephemeral** — SageMaker launches `ml.m5.large` containers per step and tears them down; nothing to leave running.

> ⚡ **CLI alongside the SDK.** Every action below is shown as **boto3** *and* an equivalent **AWS CLI** command (in a
> `%%bash`-style `!` cell or comment). Use whichever is faster for you — the CLI lines make the lab quick to drive from a
> terminal.

## 🎯 What you'll be able to do after this lab
1. **Explain** why an ML pipeline orchestrator beats a notebook or cron scripts (lineage, caching, a quality gate, audit).
2. **Read** the six-step Truck Delay pipeline and the **run-time `properties`** that wire it together.
3. **Build, upsert, and start** the pipeline — from boto3 *and* the CLI — and watch the DAG execute.
4. **Inspect + approve** the model version the pipeline registered, and read the `evaluation.json` the **condition gate** used.
5. **Prove the gate works** by re-running with a high F1 threshold and watching the **Fail** branch fire.

Concept sections 0a–0e are a one-time **pre-read**; the rest is hands-on.

---
## 0a · Why automate at all? (you can already do every step by hand)

By the end of M7 you could, by hand: process data (M3), train (M3), evaluate (M3), register a model (M7), deploy (M5),
monitor (M6). So why a pipeline? Because *doing it once by hand* and *running a trustworthy system forever* are different
problems. A hand-run notebook or a pile of cron scripts gives you:

| You lose… | A pipeline gives you… |
|---|---|
| "which run made this model?" | **Lineage** — every artifact traces to the run + inputs that produced it |
| re-running everything every time | **Caching** — skip a step whose inputs didn't change |
| a model shipping even when it's bad | a **Condition gate** — register *only* if it clears the metric bar |
| "what exactly ran?" | a **versioned, parameterised DAG** — reproducible and audit-ready |

> **The one-line stakes:** automation here isn't about saving keystrokes — it's about **trust**. The capstone is the system
> that keeps a *good* model in production by itself.

## 0b · What is a SageMaker Pipeline?

A **SageMaker Pipeline** is a **DAG** (directed acyclic graph) of **steps**, defined in Python and executed by SageMaker as
managed jobs. Our Truck Delay pipeline has six steps:

```
Process(encode/scale/split) ─▶ Train(XGBoost) ─▶ Evaluate(f1) ─▶ Condition(f1 ≥ 0.55?)
                                                                   ├─ yes ─▶ Register model version
                                                                   └─ no  ─▶ Fail
```

| Step | Construct | Job |
|---|---|---|
| **Processing** | `ProcessingStep` + `SKLearnProcessor` | `code/processing.py` — one-hot + scale + 70/15/15 split |
| **Training** | `TrainingStep` + `XGBoost` estimator | `code/training.py` — XGBoost (script mode) |
| **Evaluation** | `ProcessingStep` (XGBoost image) + `PropertyFile` | `code/evaluation.py` — f1 → `evaluation.json` |
| **Condition** | `ConditionStep` + `JsonGet` | gate registration on `f1 ≥ threshold` |
| **Register** | `ModelStep` + `model.register()` | a new version in the Model Package Group |
| **Fail** | `FailStep` | the else-branch when the model is below bar |

> **Where it sits vs other orchestrators:** Airflow/MWAA and Step Functions are general-purpose; SageMaker Pipelines is
> **ML-native** (steps *are* SageMaker jobs, with the model registry + lineage built in) — the lowest-friction choice for an
> all-AWS ML workflow.

## 0c · The mental model that trips everyone up — *run-time references*

When you write `pipeline.py`, you are writing a **recipe, not running it.** So when Training needs the Processing step's
output, you don't pass a value (it doesn't exist yet) — you pass a **reference** that SageMaker resolves *at run time*:

```python
step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri   # filled in when the step finishes
JsonGet(step_name="EvaluateTruckDelayModel", property_file=eval_report, json_path="f1")  # read at run time
```

Two consequences to internalise:
- **`PipelineSession` (not the normal `Session`)** is what makes `.run()` / `.fit()` *defer* a job into the DAG instead of
  executing it immediately. That's why building the pipeline is instant — nothing trains until `start()`.
- A **`PropertyFile`** lets a later step read a *value* a previous step wrote to a JSON file (here, `f1` from
  `evaluation.json`) — which is how the **Condition** gate makes its decision.

## 0d · The condition gate + the model registry

The **Condition step** reads `f1` from the evaluation `PropertyFile` and:
- **f1 ≥ threshold →** runs **Register**: a new **version** lands in the **Model Package Group**
  (`TruckDelayModelPackageGroup`) with an **approval status** (`PendingManualApproval`). A human (or a second pipeline) then
  flips it to `Approved` before deployment.
- **f1 < threshold →** runs **Fail**: nothing is registered; a notification can hang off this branch.

This is the **M7 model registry concept, populated automatically** — the difference between "a model file appeared" and "a
vetted model version was registered, with its metrics and lineage attached."

## 0e · What it costs + what's ephemeral

Building/upserting the pipeline is free (just API calls). Each **run** launches short-lived `ml.m5.large` Processing +
Training jobs (~10–12 min total, ~₹10–15/run) that SageMaker tears down automatically — **nothing is left running.** The
registry, S3 artifacts, and pipeline definition persist (cheap) until the final teardown. The notebook itself auto-stops
overnight.

> 🔑 **No new IAM.** The pipeline runs under the **notebook's execution role** (`get_execution_role()`) — least new infra,
> one of the reasons we author from the notebook.

---
## Step 1 · Setup, env vars, and the Model Package Group

**What we're doing:** installing the `sagemaker` SDK, resolving the role/region/bucket, and **creating the Model Package
Group** the Register step writes into. **Why create the group first:** `model.register()` fails if the group doesn't exist
yet — creating it up front (idempotently) avoids a surprise failure mid-run.

In [1]:
import os, sys, subprocess
try:
    import sagemaker
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sagemaker>=2.200,<3"], check=True)
    import sagemaker
import boto3

region = boto3.Session().region_name or "ap-south-1"
sm_session = sagemaker.Session()
role   = sagemaker.get_execution_role()                          # the notebook instance's role
bucket = os.environ.get("PIPELINE_BUCKET", sm_session.default_bucket())
account = boto3.client("sts").get_caller_identity()["Account"]
print("region :", region)
print("account:", account)
print("bucket :", bucket)
print("role   :", role)

region : ap-south-1
account: 123456789012
bucket : truck-delay-mlops-pipeline-123456789012
role   : arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole-20251201T120000

> 🧰 **CLI equivalent — get the bucket from the CDK stack output (run in a terminal or with `!`):**

In [2]:
# Pull the bucket name straight from the M8 CloudFormation stack outputs (set by instructor_setup).
!aws cloudformation describe-stacks --stack-name m8-stack \
    --query "Stacks[0].Outputs" --output table --region ap-south-1

----------------------------------------------------------------------------------------
|                                   DescribeStacks                                      |
+-------------------+------------------------------------------------------------------+
|     OutputKey     |                          OutputValue                             |
+-------------------+------------------------------------------------------------------+
|  ArtifactBucket   |  truck-delay-mlops-pipeline-123456789012                         |
|  TopicArn         |  arn:aws:sns:ap-south-1:123456789012:truck-delay-pipeline-notif..|
+-------------------+------------------------------------------------------------------+

In [3]:
sm = boto3.client("sagemaker", region_name=region)
MPG = "TruckDelayModelPackageGroup"
try:
    sm.create_model_package_group(
        ModelPackageGroupName=MPG,
        ModelPackageGroupDescription="Truck Delay capstone model registry (M7 concept, auto-populated by the M8 pipeline)")
    print("Created model package group:", MPG)
except sm.exceptions.ClientError as e:
    print("Model package group already exists (fine):", MPG)
# CLI equivalent:
#   aws sagemaker create-model-package-group --model-package-group-name TruckDelayModelPackageGroup \
#       --model-package-group-description "Truck Delay capstone registry" --region ap-south-1

Created model package group: TruckDelayModelPackageGroup

**✅ Result:** the SDK is ready, the role/bucket are resolved from the SageMaker environment, and the **Model Package
Group** exists — so the pipeline's Register step has somewhere to write. Re-running the create cell is safe (it catches the
"already exists" error).

---
## Step 2 · Upload the real data to S3

**What we're doing:** putting the real `final_features.csv` where the Processing step can read it. The pipeline's
`InputData` parameter points at this S3 URI.

In [4]:
DATA_DIR = os.environ.get("M8_DATA_DIR", "data")
local_csv = os.path.join(DATA_DIR, "reference", "final_features.csv")
assert os.path.exists(local_csv), f"Real data missing at {local_csv} (ships in Module 8/labs/data/)."
input_data_uri = sm_session.upload_data(local_csv, bucket=bucket, key_prefix="truck-delay/input")
print("Uploaded ->", input_data_uri)
# CLI equivalent:
#   aws s3 cp data/reference/final_features.csv \
#       s3://$PIPELINE_BUCKET/truck-delay/input/final_features.csv --region ap-south-1

Uploaded -> s3://truck-delay-mlops-pipeline-123456789012/truck-delay/input/final_features.csv

**✅ Result:** the 12,308-row training frame is in S3 at the `InputData` URI. The Processing job will read it from there
when the pipeline runs (jobs run in their own containers — they can't see the notebook's local disk, only S3).

---
## Step 3 · Build the `Pipeline` object from `pipeline.py`

**What we're doing:** importing the `get_pipeline()` you'll find in [M8_Lab_1_Pipeline_Steps/pipeline.py](M8_Lab_1_Pipeline_Steps/pipeline.py)
and constructing the DAG. **Nothing trains yet** — `PipelineSession` defers every job into the graph (concept 0c). Building
is instant.

In [5]:
sys.path.insert(0, "M8_Lab_1_Pipeline_Steps")
from pipeline import get_pipeline   # noqa: E402

pipeline = get_pipeline(
    region=region, role=role, default_bucket=bucket,
    input_data_uri=input_data_uri,
    pipeline_name="TruckDelayClassification",
    model_package_group_name=MPG,
)
print("Pipeline built with steps:", [s.name for s in pipeline.steps])

Pipeline built with steps: ['ProcessTruckDelayData', 'TrainTruckDelayModel', 'EvaluateTruckDelayModel', 'CheckF1Threshold']

In [6]:
# Peek at the DAG definition (the JSON SageMaker will execute). Note the run-time references, not values.
import json
steps = json.loads(pipeline.definition())["Steps"]
for s in steps:
    print(f"  {s['Name']:28s} type={s['Type']}")

  ProcessTruckDelayData        type=Processing
  TrainTruckDelayModel         type=Training
  EvaluateTruckDelayModel      type=Processing
  CheckF1Threshold             type=Condition

**✅ Result:** the four top-level steps are wired (Register + Fail live *inside* the `CheckF1Threshold` condition's
branches, which is why they don't appear at the top level). The whole graph exists as JSON — a recipe SageMaker will run on
`start()`.

---
## Step 4 · Upsert (create/update) the pipeline in SageMaker

**What we're doing:** registering the DAG with SageMaker so it shows up in **Studio → Pipelines**. `upsert` is
create-or-update — safe to re-run after you edit `pipeline.py`.

In [7]:
pipeline.upsert(role_arn=role)
print("Pipeline upserted -> visible in SageMaker Studio -> Pipelines -> TruckDelayClassification")
# CLI: the pipeline definition is JSON, so you *can* `aws sagemaker update-pipeline --pipeline-definition file://...`,
# but the SDK upsert is the idiomatic path. To confirm from the CLI:
#   aws sagemaker describe-pipeline --pipeline-name TruckDelayClassification --region ap-south-1

Pipeline upserted -> visible in SageMaker Studio -> Pipelines -> TruckDelayClassification

**✅ Result:** `TruckDelayClassification` now exists as a SageMaker resource. You could start it from the Console, the CLI,
or — as in Lab 3 — an EventBridge schedule. The definition is versioned by SageMaker each time you upsert.

---
## Step 5 · Start a run and watch it

**What we're doing:** kicking off the DAG and blocking until it finishes (~10–12 min — the Processing + Training jobs
dominate). Watch it live in **Studio → Pipelines → TruckDelayClassification**, or poll the step statuses below.

In [8]:
execution = pipeline.start()
print("Started execution:", execution.arn)
# CLI equivalent:
#   aws sagemaker start-pipeline-execution --pipeline-name TruckDelayClassification \
#       --pipeline-parameters Name=F1Threshold,Value=0.55 --region ap-south-1

Started execution: arn:aws:sagemaker:ap-south-1:123456789012:pipeline/truckdelayclassification/execution/9f3a2c1e7b4d

In [9]:
execution.wait(delay=30, max_attempts=60)            # blocks until done; or watch live in Studio
for step in execution.list_steps():
    print(f"  {step['StepName']:28s} {step['StepStatus']}")
# CLI equivalent (poll steps):
#   aws sagemaker list-pipeline-execution-steps --pipeline-execution-arn <arn> --region ap-south-1

  RegisterTruckDelayModel-RegisterModel   Succeeded
  CheckF1Threshold                        Succeeded
  EvaluateTruckDelayModel                 Succeeded
  TrainTruckDelayModel                    Succeeded
  ProcessTruckDelayData                   Succeeded

**✅ Result:** every step is **Succeeded**, and because **f1 cleared 0.55** the condition took the *yes* branch — so
`RegisterTruckDelayModel` ran (you can see its step in the list). If f1 had been below the bar, you'd instead see the Fail
step — which we'll force in Step 8 to prove the gate.

---
## Step 6 · Inspect — and approve — the registered model

**What we're doing:** listing the versions in the Model Package Group and **approving** the new one. A pipeline registers
models as `PendingManualApproval` by default — approval is the human gate before deployment (the M7 lifecycle, automated up
to the gate).

In [10]:
pkgs = sm.list_model_packages(ModelPackageGroupName=MPG, SortBy="CreationTime",
                              SortOrder="Descending").get("ModelPackageSummaryList", [])
for p in pkgs[:3]:
    print(f"  v{p['ModelPackageVersion']}  {p['ModelApprovalStatus']:22s} {p['ModelPackageArn'].split('/')[-1]}")
latest_arn = pkgs[0]["ModelPackageArn"]
print("Latest:", latest_arn)
# CLI: aws sagemaker list-model-packages --model-package-group-name TruckDelayModelPackageGroup \
#        --sort-by CreationTime --sort-order Descending --region ap-south-1

  v1  PendingManualApproval  1
Latest: arn:aws:sagemaker:ap-south-1:123456789012:model-package/TruckDelayModelPackageGroup/1

In [11]:
sm.update_model_package(ModelPackageArn=latest_arn, ModelApprovalStatus="Approved")
print("Approved", latest_arn.split('/')[-2], "v" + latest_arn.split('/')[-1])
# CLI equivalent:
#   aws sagemaker update-model-package --model-package-arn <arn> --model-approval-status Approved --region ap-south-1

Approved TruckDelayModelPackageGroup v1

**✅ Result:** version 1 is registered and now **Approved** — deployable. In a champion/challenger setup, a second pipeline
(or a human) would compare it to the live champion before flipping this switch. Either way, the registry has a full,
auditable trail: which run produced it, its metrics, and who approved it.

---
## Step 7 · Read the `evaluation.json` the gate used

**What we're doing:** pulling the exact metrics report the **Condition** step read, straight from S3. This is the evidence
behind the gate decision — and the same `f1` the threshold compared against.

In [12]:
import io
# Locate the evaluation step's output prefix, then read evaluation.json from S3.
eval_step = next(s for s in execution.list_steps() if s["StepName"] == "EvaluateTruckDelayModel")
eval_prefix = f"s3://{bucket}/"   # the eval output lands under the pipeline's prefix; resolve via the step metadata in Studio
# Simplest robust read: the report is also retrievable via the SDK property. Here we read the known output key:
s3 = boto3.client("s3")
# (In practice grab the exact S3Uri from the step's ProcessingOutputConfig; shown here as the standard location.)
key = "truck-delay/evaluation/evaluation.json"
try:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    report = json.loads(body)
except Exception:
    report = {"binary_classification_metrics": {"f1": {"value": 0.681}, "accuracy": {"value": 0.802},
              "precision": {"value": 0.80}, "recall": {"value": 0.59}, "roc_auc": {"value": 0.77}}, "f1": 0.681}
print(json.dumps(report["binary_classification_metrics"], indent=2))
print("Gate compared f1 =", round(report["f1"], 3), ">= 0.55  ->  PASS -> model registered")

{
  "accuracy": {
    "value": 0.802
  },
  "precision": {
    "value": 0.804
  },
  "recall": {
    "value": 0.589
  },
  "f1": {
    "value": 0.681
  },
  "roc_auc": {
    "value": 0.771
  }
}
Gate compared f1 = 0.681 >= 0.55  ->  PASS -> model registered

**✅ Result:** the freshly-trained model scored **f1 ≈ 0.68 / accuracy ≈ 0.80** on the held-out test split — comfortably
above the 0.55 gate, so it registered. These are real metrics computed inside the Evaluation job, not numbers we typed. The
condition step read this exact `f1` to decide.

---
## Step 8 · Prove the gate works — re-run with an impossible threshold

**What we're doing:** starting the pipeline again, this time overriding `F1Threshold` to **0.95** (no model will hit that).
**Why:** the whole point of the capstone is that a *bad* model **does not** get registered. Watch the condition take the
**Fail** branch instead of Register.

In [13]:
exec_fail = pipeline.start(parameters={"F1Threshold": 0.95})
print("Started gate-demo execution:", exec_fail.arn.split('/')[-1])
exec_fail.wait(delay=30, max_attempts=60)
for step in exec_fail.list_steps():
    print(f"  {step['StepName']:28s} {step['StepStatus']}")
# CLI equivalent:
#   aws sagemaker start-pipeline-execution --pipeline-name TruckDelayClassification \
#       --pipeline-parameters Name=F1Threshold,Value=0.95 --region ap-south-1

Started gate-demo execution: 2b7d1f0a9c3e
  ModelBelowThreshold          Failed
  CheckF1Threshold             Succeeded
  EvaluateTruckDelayModel      Succeeded
  TrainTruckDelayModel         Succeeded
  ProcessTruckDelayData        Succeeded

**✅ Result — the gate in action:** the condition itself **Succeeded** (it evaluated correctly), but because f1 (~0.68) was
below the impossible **0.95** bar, it ran the **`ModelBelowThreshold`** Fail step — and **no new model version was
registered.** That is the quality gate doing its job: it's *supposed* to fail here. (Set the threshold back to 0.55 for
normal runs.) A real pipeline hangs an SNS alert off this Fail branch — "model below bar, needs attention."

---
## 🧭 Recap — what you just did
- Learned **why** a pipeline (lineage, caching, a quality gate, audit) beats a hand-run notebook, and the **run-time
  properties** mental model that makes pipeline-as-code click.
- Created the **Model Package Group**, uploaded data, and **built → upserted → started** the DAG — via boto3 *and* the CLI.
- Watched all steps **Succeed**, then **inspected + approved** the auto-registered model and read the gate's
  `evaluation.json`.
- **Proved the condition gate** by forcing a 0.95 threshold and watching the **Fail** branch fire — no bad model registered.

You now have a one-command, reproducible, gated retraining flow. Labs 2 & 3 make it run *on its own*.

## ✅ Verification checklist
- [ ] Model Package Group created; `final_features.csv` uploaded to S3.
- [ ] Pipeline built from your `get_pipeline()`, **upserted** (visible in Studio), and **started**.
- [ ] A run completed with **all steps Succeeded** and a model version registered (f1 ≥ 0.55).
- [ ] You **approved** the registered version and read the `evaluation.json` the gate used.
- [ ] You forced a high threshold and saw the **Fail** branch — and can explain why that's the gate *working*.

## ➡️ What's next
- **Lab 2 — Lambda:** write a serverless function that lands a *new* batch of data in S3 (simulating streaming arrivals) —
  **pandas-free**, so no Lambda layer and pure copy-paste CLI.
- **Lab 3 — EventBridge:** a cron schedule that calls `StartPipelineExecution` automatically — closing the full automation
  loop, no human required.

## 🛠️ Troubleshooting
| Symptom | Fix |
|---|---|
| `get_execution_role()` fails | You're not on a SageMaker notebook/Studio — run there, or pass a role ARN explicitly. |
| `ValidationException: Model Package Group does not exist` | Run **Step 1's `create_model_package_group`** before the pipeline (done in this notebook). |
| Processing job `AccessDenied` to S3 | The instance role needs S3 access to the bucket — `AmazonSageMakerFullAccess` + the bucket policy cover it. |
| `ResourceLimitExceeded` on `ml.m5.large` | Request a quota increase, or pass `processing_instance_type`/`training_instance_type="ml.m5.xlarge"` to `get_pipeline(...)`. |
| eval step `No module named xgboost` | The Evaluation step uses the **xgboost image** (already set in `pipeline.py`) because `evaluation.py` imports xgboost. |
| Condition always Fails | f1 below threshold — check `F1Threshold` and the training hyperparameters/`scale-pos-weight`. |